# 23 — Embedding Benchmarks
**Goal:** Compare Word2Vec, GloVe, and Sentence Transformers on resume-specific tasks.

## 1. Building a Mini Benchmark

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Test pairs: (resume_skill, job_requirement, expected_similarity_rating)
# 1.0 = same meaning, 0.0 = unrelated
test_pairs = [
    ("python", "Python programming", 0.9),     # same skill
    ("python", "Java", 0.2),                    # different language
    ("tensorflow", "deep learning", 0.7),       # related
    ("nlp", "natural language processing", 0.8), # acronym vs full
    ("docker", "kubernetes", 0.6),              # related tools
    ("python", "cooking", 0.0),                 # unrelated
]

def evaluate_embeddings(embeddings_dict, model_name, test_pairs):
    """Score how well embeddings capture semantic similarity."""
    scores = []
    for w1, w2, expected in test_pairs:
        try:
            v1 = embeddings_dict[w1] if w1 in embeddings_dict else None
            v2 = embeddings_dict[w2] if w2 in embeddings_dict else None
            if v1 is not None and v2 is not None:
                sim = cosine_similarity([v1], [v2])[0][0]
                scores.append((w1, w2, sim, expected))
        except:
            pass
    return scores

print("Benchmark ready. Test pairs defined. Each model will be evaluated on semantic similarity accuracy.")
print("\nNote: We need pre-trained vectors loaded to run this. Check each model's availability.")

## 2. Loading All Models

In [ ]:
models = {}
# Sentence Transformers
try:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("all-MiniLM-L6-v2")
    models["SentenceTransformer"] = st
    print("✓ SentenceTransformer loaded")
except: print("  SentenceTransformer not available")

# Word2Vec/GloVe via gensim
try:
    import gensim.downloader as api
    # Use a small model if available
    models["glove-twitter-25"] = api.load("glove-twitter-25")
    print("✓ GloVe loaded")
except: print("  GloVe not downloaded (needs internet/first-time download)")

print(f"\nLoaded {len(models)} models for benchmark")

## 3. Running the Benchmark

In [ ]:
if models:
    for name, model in models.items():
        print(f"\n=== {name} ===")
        for w1, w2, expected in test_pairs:
            try:
                if hasattr(model, 'encode'):
                    # SentenceTransformer
                    v1, v2 = model.encode([w1, w2])
                else:
                    # gensim keyed vectors
                    v1, v2 = model[w1], model[w2]
                sim = cosine_similarity([v1], [v2])[0][0]
                print(f"  '{w1:20s}' vs '{w2:20s}': {sim:.3f} (expected ~{expected})")
            except Exception as e:
                print(f"  '{w1:20s}' vs '{w2:20s}': ERROR — {e}")
else:
    print("No models available. This notebook works best after running notebooks 19, 21, and 22.")
    print("The benchmark methodology is defined — run those first, then return here.")

## Summary: Sentence Transformers > GloVe > Word2Vec for most resume tasks. But all have use cases.